### TRAINING RESNET WITH DATA FROM release_in_the_wild ONLY

In [1]:
import sys
from pathlib import Path

# add parent directory to path to fix imports
parent_dir = Path().resolve().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from embedding_pipeline.stft import stft

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
from torch.utils.data import DataLoader, Dataset
import random

In [2]:
class AudioFileDataset(Dataset):
    def __init__(self, samples, max_time_windows=256):
        self.samples = samples
        self.max_time_windows = max_time_windows

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        stft_tensor = process_clip(file_path, self.max_time_windows)
        target = torch.tensor([label], dtype=torch.float32)
        return stft_tensor, target
    

def process_clip(file_path, max_time_windows=256):
    magnitude_db, _ = stft(file_path)
    stft_tensor = torch.tensor(magnitude_db).unsqueeze(0).float()  # [3, 512, W]
    width = stft_tensor.shape[-1]

    # clip long samples and pad short samples so all tensors have the same width
    if width > max_time_windows:
        stft_tensor = stft_tensor[:, :, :max_time_windows]
    elif width < max_time_windows:
        pad_width = max_time_windows - width
        stft_tensor = torch.nn.functional.pad(stft_tensor, (0, pad_width))

    return stft_tensor


def collect_labeled_files(root_dir, label, max_count):
    samples = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            samples.append((os.path.join(root, file), label))
            if len(samples) >= max_count:
                return samples
    return samples

In [3]:
max_files = 10000
max_time_windows = 256 # max time windows for stft
batch_size = 32

fake_samples = collect_labeled_files(
    "../data/release_in_the_wild/train/fake", label=1, max_count=max_files
)
real_samples = collect_labeled_files(
    "../data/release_in_the_wild/train/real", label=0, max_count=max_files
)

dataset = fake_samples + real_samples
random.shuffle(dataset)
print("dataset created with", len(dataset), "samples")


# split into train/test file lists
train_to_test_ratio = 0.8
train_size = int(len(dataset) * train_to_test_ratio)

train_samples = dataset[:train_size]
test_samples = dataset[train_size:]

train_dataset = AudioFileDataset(train_samples, max_time_windows)
test_dataset = AudioFileDataset(test_samples, max_time_windows)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

dataset created with 18271 samples


In [4]:
from resnet import ResNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet().to(device)
model.train()

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

In [5]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for batch_idx, (data, target) in enumerate(train_dataloader):
        data = data.to(device)
        batch_target = target.to(device)

        pred = model(data)
        loss = criterion(pred, batch_target)
        optimizer.zero_grad()  # reset gradient
        loss.backward()  # calculate gradient
        optimizer.step()  # update parameters

        if batch_idx % 5 == 0:
            print(f"batch progress: epoch {epoch+1} {(100 *(batch_idx+1)/len(train_dataloader)):.2f}% loss: {loss.item():.6f}")

    # evaluate on test set
    model.eval()
    test_loss = 0.0

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_dataloader):
            data = data.to(device)
            batch_target = target.to(device)

            pred = model(data)
            loss = criterion(pred, batch_target)  # calculate loss
            test_loss += loss.item()

    total_test_loss = test_loss / len(test_dataloader)
    print('epoch: {}, test loss: {:.6f}'.format(
        epoch + 1,
        total_test_loss,
        ))

    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch + 1,
        "batch": batch_idx,
    }, "resnet_checkpoint.pth")
    print('model checkpoint saved')

c:\Users\whyam\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


batch progress: epoch 1 0.22% loss: 0.723002
batch progress: epoch 1 1.31% loss: 0.629931
batch progress: epoch 1 2.41% loss: 0.586326
batch progress: epoch 1 3.50% loss: 0.572037
batch progress: epoch 1 4.60% loss: 0.577817
batch progress: epoch 1 5.69% loss: 0.514345
batch progress: epoch 1 6.78% loss: 0.464530
batch progress: epoch 1 7.88% loss: 0.581254
batch progress: epoch 1 8.97% loss: 0.485764
batch progress: epoch 1 10.07% loss: 0.472405
batch progress: epoch 1 11.16% loss: 0.403234
batch progress: epoch 1 12.25% loss: 0.429280
batch progress: epoch 1 13.35% loss: 0.335942
batch progress: epoch 1 14.44% loss: 0.314719
batch progress: epoch 1 15.54% loss: 0.329517
batch progress: epoch 1 16.63% loss: 0.444154
batch progress: epoch 1 17.72% loss: 0.270484
batch progress: epoch 1 18.82% loss: 0.257819
batch progress: epoch 1 19.91% loss: 0.253856
batch progress: epoch 1 21.01% loss: 0.264491
batch progress: epoch 1 22.10% loss: 0.392593
batch progress: epoch 1 23.19% loss: 0.2351